In [16]:
# =========================
# CELL 1: INSTALLS
# =========================
import sys
import subprocess
import importlib

def ensure_install(package_name, import_name=None):
    name = import_name or package_name
    try:
        importlib.import_module(name)
        print(f"OK: {package_name}")
    except Exception:
        print(f"Installing: {package_name}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])

ensure_install("transformers")
ensure_install("datasets")
ensure_install("peft")
ensure_install("accelerate")
ensure_install("bitsandbytes")
ensure_install("scikit-learn", "sklearn")
ensure_install("lxml")
ensure_install("sentencepiece")
ensure_install("pandas")
ensure_install("numpy")

OK: transformers
OK: datasets
OK: peft
OK: accelerate
OK: bitsandbytes
OK: scikit-learn
OK: lxml
OK: sentencepiece
OK: pandas
OK: numpy


In [17]:
# =========================
# CELL 2: IMPORTS
# =========================
import os
import re
import gc
import math
import time
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from lxml import etree as ET
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Torch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


In [18]:
# =========================
# CELL 3: FIND FILES
# =========================
SEARCH_ROOTS = ["/content", "/kaggle/input", "/kaggle/working"]

def find_file(filename, search_roots):
    matches = []
    for root in search_roots:
        if os.path.exists(root):
            for dirpath, dirnames, filenames in os.walk(root):
                if filename in filenames:
                    matches.append(os.path.join(dirpath, filename))
    return matches

train_matches = find_file("train.csv", SEARCH_ROOTS)
test_matches = find_file("test.csv", SEARCH_ROOTS)
sub_matches = find_file("sample_submission.csv", SEARCH_ROOTS)

print("train.csv matches:")
for x in train_matches:
    print(" ", x)

print("\ntest.csv matches:")
for x in test_matches:
    print(" ", x)

print("\nsample_submission.csv matches:")
for x in sub_matches:
    print(" ", x)

if len(train_matches) == 0:
    raise FileNotFoundError("Could not find train.csv")
if len(test_matches) == 0:
    raise FileNotFoundError("Could not find test.csv")
if len(sub_matches) == 0:
    raise FileNotFoundError("Could not find sample_submission.csv")

TRAIN_PATH = train_matches[0]
TEST_PATH = test_matches[0]
SAMPLE_SUB_PATH = sub_matches[0]

OUTPUT_DIR = "/content/lora_svg_outputs"
SUB_PATH = "/content/submission.csv"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("\nUsing:")
print("TRAIN_PATH =", TRAIN_PATH)
print("TEST_PATH =", TEST_PATH)
print("SAMPLE_SUB_PATH =", SAMPLE_SUB_PATH)
print("SUB_PATH =", SUB_PATH)

train.csv matches:
  /content/train.csv

test.csv matches:
  /content/test.csv

sample_submission.csv matches:
  /content/sample_submission.csv

Using:
TRAIN_PATH = /content/train.csv
TEST_PATH = /content/test.csv
SAMPLE_SUB_PATH = /content/sample_submission.csv
SUB_PATH = /content/submission.csv


In [19]:
# =========================
# CELL 4: CONFIG
# =========================
# <= 2B model, instructor-compliant
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

# Use only provided competition data
USE_4BIT = torch.cuda.is_available()

# Training subset size
# Increase if you have stronger GPU / more time
MAX_TRAIN_SAMPLES = 8000
MAX_VAL_SAMPLES = 500

# Token lengths
MAX_SEQ_LENGTH = 512
MAX_NEW_TOKENS = 180

# LoRA
LORA_R = 16
LORA_ALPHA = 16
LORA_DROPOUT = 0.05

# Training
NUM_EPOCHS = 1
PER_DEVICE_BATCH = 1
GRAD_ACC = 4
LEARNING_RATE = 2e-4
WARMUP_STEPS = 30
WEIGHT_DECAY = 0.01
LOGGING_STEPS = 10

# Inference
BATCH_SIZE_INFER = 8

# SVG constraints
MAX_SVG_CHARS = 16000

SYSTEM_PROMPT = (
    "Generate a compact valid SVG from the user prompt. "
    "Return only SVG code. "
    "Use a single root <svg> element with width='256' height='256' viewBox='0 0 256 256'. "
    "Keep the SVG short and visually faithful."
)

FALLBACK_SVG = """<svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256">
<rect x="0" y="0" width="256" height="256" fill="white"/>
<circle cx="128" cy="128" r="60" fill="none" stroke="black" stroke-width="8"/>
</svg>"""

In [20]:
# =========================
# CELL 5: LOAD DATA
# =========================
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
sample_sub = pd.read_csv(SAMPLE_SUB_PATH)

print("train shape:", train_df.shape)
print("test shape :", test_df.shape)
print("sample shape:", sample_sub.shape)

print("\ntrain columns:", train_df.columns.tolist())
print("test columns :", test_df.columns.tolist())
print("sample columns:", sample_sub.columns.tolist())

assert {"id", "prompt", "svg"}.issubset(train_df.columns), "train.csv must contain id, prompt, svg"
assert {"id", "prompt"}.issubset(test_df.columns), "test.csv must contain id, prompt"
assert "id" in sample_sub.columns, "sample_submission.csv must contain id"

train_df["prompt"] = train_df["prompt"].fillna("").astype(str)
train_df["svg"] = train_df["svg"].fillna("").astype(str)
test_df["prompt"] = test_df["prompt"].fillna("").astype(str)

print("\nSample prompts:")
print(train_df["prompt"].head(3).tolist())

train shape: (50000, 3)
test shape : (1000, 2)
sample shape: (1000, 2)

train columns: ['id', 'prompt', 'svg']
test columns : ['id', 'prompt']
sample columns: ['id', 'svg']

Sample prompts:
['The image features two orange squares with a microphone icon and an arrow connecting them, set against a white background.', 'A simple smiley face with a wide open mouth and straight eyes.', 'The image features a black-outlined icon of a camera against a white background. The camera has a rectangular shape with rounded edges, a large circular lens in the center, and a smaller oval-shaped button on the top right.']


In [21]:
# =========================
# CELL 6: SVG HELPERS
# =========================
def simple_extract_svg(text: str):
    if not isinstance(text, str):
        return None

    text = text.strip()
    text = text.replace("```svg", "").replace("```xml", "").replace("```", "").strip()

    m = re.search(r"(<svg[\s\S]*?</svg>)", text, flags=re.I)
    if m:
        return m.group(1).strip()

    low = text.lower()
    if "<svg" in low and "</svg>" in low:
        start = low.find("<svg")
        end = low.rfind("</svg>")
        if start != -1 and end != -1:
            return text[start:end+6].strip()

    return None

def simple_repair_svg(text: str):
    svg = simple_extract_svg(text)
    if svg is None:
        return FALLBACK_SVG

    svg = re.sub(r"^\s*<\?xml[^>]*\?>", "", svg, flags=re.I)
    svg = re.sub(r"<!DOCTYPE[^>]*>", "", svg, flags=re.I)
    svg = re.sub(r"<!--.*?-->", "", svg, flags=re.S)
    svg = re.sub(r">\s+<", "><", svg)
    svg = re.sub(r"\s+", " ", svg).strip()

    if 'xmlns=' not in svg:
        svg = svg.replace("<svg", '<svg xmlns="http://www.w3.org/2000/svg"', 1)
    if 'width=' not in svg:
        svg = svg.replace("<svg", '<svg width="256"', 1)
    if 'height=' not in svg:
        svg = svg.replace("<svg", '<svg height="256"', 1)
    if 'viewBox=' not in svg:
        svg = svg.replace("<svg", '<svg viewBox="0 0 256 256"', 1)

    if len(svg) > MAX_SVG_CHARS:
        return FALLBACK_SVG

    return svg

def simple_is_valid_svg(svg: str):
    try:
        if not isinstance(svg, str):
            return False
        svg = svg.strip()
        if not svg.lower().startswith("<svg"):
            return False
        if "</svg>" not in svg.lower():
            return False
        if len(svg) > MAX_SVG_CHARS:
            return False
        ET.fromstring(svg.encode("utf-8"))
        return True
    except Exception:
        return False

def light_clean_svg(svg):
    if not isinstance(svg, str):
        return None
    svg = svg.strip()
    if len(svg) == 0:
        return None
    if "<svg" not in svg.lower() or "</svg>" not in svg.lower():
        return None

    svg = simple_repair_svg(svg)
    if simple_is_valid_svg(svg):
        return svg
    return None

print("Fallback valid:", simple_is_valid_svg(FALLBACK_SVG))

Fallback valid: True


In [22]:
# =========================
# CELL 7: CLEAN + SAMPLE PROVIDED DATA ONLY
# =========================
train_df["svg_clean"] = train_df["svg"].map(light_clean_svg)

usable = train_df["svg_clean"].notna().sum()
print("Usable rows before filtering:", usable, "/", len(train_df))

train_df = train_df[train_df["svg_clean"].notna()].copy().reset_index(drop=True)
train_df = train_df[train_df["prompt"].str.strip() != ""].copy().reset_index(drop=True)

train_df["prompt_len"] = train_df["prompt"].str.split().str.len()
train_df["svg_len"] = train_df["svg_clean"].str.len()

print("Usable rows after filtering:", len(train_df))

# balanced subset sampling for feasibility
if len(train_df) > MAX_TRAIN_SAMPLES + MAX_VAL_SAMPLES:
    train_df["prompt_bin"] = pd.qcut(train_df["prompt_len"], q=5, duplicates="drop")
    train_df["svg_bin"] = pd.qcut(train_df["svg_len"], q=5, duplicates="drop")
    train_df["bucket"] = train_df["prompt_bin"].astype(str) + " || " + train_df["svg_bin"].astype(str)

    sampled_parts = []
    per_bucket = max(1, (MAX_TRAIN_SAMPLES + MAX_VAL_SAMPLES) // train_df["bucket"].nunique())

    for _, grp in train_df.groupby("bucket"):
        take = min(len(grp), per_bucket)
        sampled_parts.append(grp.sample(n=take, random_state=SEED))

    train_df = pd.concat(sampled_parts, axis=0).drop_duplicates(subset=["id"]).reset_index(drop=True)

    # if still too large, trim
    if len(train_df) > MAX_TRAIN_SAMPLES + MAX_VAL_SAMPLES:
        train_df = train_df.sample(n=MAX_TRAIN_SAMPLES + MAX_VAL_SAMPLES, random_state=SEED).reset_index(drop=True)

print("Rows after subset sampling:", len(train_df))

train_df = train_df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

val_size = min(MAX_VAL_SAMPLES, max(1, int(0.05 * len(train_df))))
val_df = train_df.iloc[:val_size].copy().reset_index(drop=True)
trn_df = train_df.iloc[val_size:].copy().reset_index(drop=True)

if len(trn_df) > MAX_TRAIN_SAMPLES:
    trn_df = trn_df.iloc[:MAX_TRAIN_SAMPLES].copy().reset_index(drop=True)

print("Train rows:", len(trn_df))
print("Val rows  :", len(val_df))

Usable rows before filtering: 50000 / 50000
Usable rows after filtering: 50000
Rows after subset sampling: 8500
Train rows: 8000
Val rows  : 425


In [23]:
# =========================
# CELL 8: OPTIONAL RETRIEVAL FALLBACK
# =========================
# This is NOT the main method.
# It is only a lightweight fallback if the LoRA model produces malformed SVG.

vectorizer = TfidfVectorizer(
    lowercase=True,
    strip_accents="unicode",
    ngram_range=(1, 2),
    min_df=1,
    max_features=50000,
    token_pattern=r"(?u)\b\w+\b",
)

X_train_prompts = vectorizer.fit_transform(trn_df["prompt"].tolist())

def retrieve_fallback_svg(prompt: str, k: int = 1):
    q = vectorizer.transform([str(prompt)])
    sims = linear_kernel(q, X_train_prompts).ravel()
    idx = int(np.argmax(sims))
    return simple_repair_svg(trn_df.iloc[idx]["svg_clean"])

print("Fallback retrieval ready.")

Fallback retrieval ready.


In [24]:
# =========================
# CELL 9: FORMAT TRAINING TEXT
# =========================
def make_train_text(prompt: str, svg: str) -> str:
    # short format for faster tokenization/training
    return (
        f"System: {SYSTEM_PROMPT}\n"
        f"User: {prompt}\n"
        f"Assistant: {svg}"
    )

train_texts = [make_train_text(p, s) for p, s in zip(trn_df["prompt"], trn_df["svg_clean"])]
val_texts = [make_train_text(p, s) for p, s in zip(val_df["prompt"], val_df["svg_clean"])]

train_ds = Dataset.from_dict({"text": train_texts})
val_ds = Dataset.from_dict({"text": val_texts})

print("Train samples:", len(train_ds))
print("Val samples  :", len(val_ds))

Train samples: 8000
Val samples  : 425


In [25]:
# =========================
# CELL 10: TOKENIZER + MODEL (FIXED)
# =========================
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model

# -------------------------
# 1. Load tokenizer
# -------------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

# 🔥 CRITICAL FIX (VERY IMPORTANT)
# For decoder-only models like Qwen
tokenizer.padding_side = "left"

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Tokenizer loaded.")
print("Padding side:", tokenizer.padding_side)

# -------------------------
# 2. Load model (4-bit if possible)
# -------------------------
model = None

if USE_4BIT and torch.cuda.is_available():
    try:
        print("Trying 4-bit load...")

        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.float16,
        )

        model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME,
            quantization_config=bnb_config,
            torch_dtype=torch.float16,
            device_map="auto",
            trust_remote_code=True,
        )

        model = prepare_model_for_kbit_training(model)
        print("✅ Loaded 4-bit model.")

    except Exception as e:
        print("⚠️ 4-bit load failed:", e)
        model = None

# -------------------------
# 3. Fallback (full precision)
# -------------------------
if model is None:
    print("Falling back to standard load...")

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map="auto" if torch.cuda.is_available() else None,
        trust_remote_code=True,
    )

# -------------------------
# 4. Apply LoRA
# -------------------------
peft_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "up_proj", "down_proj", "gate_proj"
    ],
)

model = get_peft_model(model, peft_config)

# -------------------------
# 5. Print trainable params
# -------------------------
model.print_trainable_parameters()

print("Model + LoRA ready.")

Tokenizer loaded.
Padding side: left
Trying 4-bit load...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

✅ Loaded 4-bit model.
trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820
Model + LoRA ready.


In [26]:
# =========================
# CELL 11: TOKENIZE DATA
# =========================
def tokenize_fn(batch):
    out = tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_SEQ_LENGTH,
    )
    out["labels"] = out["input_ids"].copy()
    return out

train_tok = train_ds.map(tokenize_fn, batched=True, remove_columns=["text"])
val_tok = val_ds.map(tokenize_fn, batched=True, remove_columns=["text"])

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

print("Tokenized train:", len(train_tok))
print("Tokenized val  :", len(val_tok))

Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

Map:   0%|          | 0/425 [00:00<?, ? examples/s]

Tokenized train: 8000
Tokenized val  : 425


In [27]:
# =========================
# CELL 12: LORA FINE-TUNING
# =========================
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_BATCH,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRAD_ACC,
    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    weight_decay=WEIGHT_DECAY,
    logging_steps=LOGGING_STEPS,
    save_strategy="epoch",
    eval_strategy="epoch",
    report_to="none",
    fp16=torch.cuda.is_available(),
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    data_collator=data_collator,
)

print("Starting LoRA fine-tuning...")
trainer.train()

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("LoRA fine-tuning complete.")

Starting LoRA fine-tuning...


Epoch,Training Loss,Validation Loss
1,0.389773,0.430438


LoRA fine-tuning complete.


In [28]:
# =========================
# CELL 13: INFERENCE HELPERS
# =========================
model.eval()

def make_infer_prompt(prompt: str) -> str:
    # keep prompt short for faster inference, as instructor suggested
    return (
        f"{SYSTEM_PROMPT}\n"
        f"User: {prompt}\n"
        f"Assistant:"
    )

def batched_generate_svg(prompts, batch_size=BATCH_SIZE_INFER):
    all_svgs = []

    with torch.no_grad():
        for start in range(0, len(prompts), batch_size):
            batch_prompts = prompts[start:start+batch_size]
            texts = [make_infer_prompt(p) for p in batch_prompts]

            inputs = tokenizer(
                texts,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=256,
            )

            if torch.cuda.is_available():
                inputs = {k: v.to(model.device) for k, v in inputs.items()}

            outputs = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                num_beams=1,
                use_cache=True,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )

            input_lens = inputs["input_ids"].shape[1]

            for i in range(outputs.shape[0]):
                gen = tokenizer.decode(outputs[i][input_lens:], skip_special_tokens=True)
                svg = simple_repair_svg(gen)

                # fallback only if malformed
                if not simple_is_valid_svg(svg):
                    svg = retrieve_fallback_svg(batch_prompts[i], k=1)

                svg = simple_repair_svg(svg)
                all_svgs.append(svg)

            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    return all_svgs

In [29]:
# =========================
# CELL 14: QUICK SANITY CHECK
# =========================
sample_prompts = val_df["prompt"].head(5).tolist()
sample_preds = batched_generate_svg(sample_prompts, batch_size=2)

for i, (p, s) in enumerate(zip(sample_prompts, sample_preds), 1):
    print("=" * 80)
    print("PROMPT:", p)
    print("VALID :", simple_is_valid_svg(s))
    print("LEN   :", len(s))
    print(s[:300])

PROMPT: A cartoon-style oven mitt icon featuring a light gray body, dark gray outlines, and a pink wrist strap with vertical lines.
VALID : True
LEN   : 228
<svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256"><rect x="0" y="0" width="256" height="256" fill="white"/><circle cx="128" cy="128" r="60" fill="none" stroke="black" stroke-width="8"/></svg>
PROMPT: The image shows one black checkmark symbol centered on a white background. The checkmark consists of two thick black lines forming a triangular shape, with no additional elements or textures present.
VALID : True
LEN   : 228
<svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256"><rect x="0" y="0" width="256" height="256" fill="white"/><circle cx="128" cy="128" r="60" fill="none" stroke="black" stroke-width="8"/></svg>
PROMPT: The image contains a black outline of a rectangular shape with two circular elements: one smaller circle near the top center and another slightl

In [30]:
# =========================
# CELL 15: BUILD SUBMISSION
# =========================
test_prompts = test_df["prompt"].tolist()

start = time.time()
pred_svgs = batched_generate_svg(test_prompts, batch_size=BATCH_SIZE_INFER)
elapsed = (time.time() - start) / 60

print(f"Inference finished in {elapsed:.2f} minutes")

submission = pd.DataFrame({
    "id": test_df["id"],
    "svg": pred_svgs
})

submission["svg"] = submission["svg"].map(simple_repair_svg)

submission = sample_sub[["id"]].merge(submission, on="id", how="left")
submission["svg"] = submission["svg"].fillna(FALLBACK_SVG)

valid_rate = submission["svg"].map(simple_is_valid_svg).mean()
print("VALID RATE:", valid_rate)

submission.to_csv(SUB_PATH, index=False)
print("Saved:", SUB_PATH)

submission.head()

Inference finished in 57.32 minutes
VALID RATE: 1.0
Saved: /content/submission.csv


,id,svg
0,fa1d8fa7-080f-4269-a9cf-a17562c9a0ca,"<svg xmlns=""http://www.w3.org/2000/svg"" width=..."
1,6eede943219547c22ac56085027d33cc,"<svg xmlns=""http://www.w3.org/2000/svg"" width=..."
2,ea045c7a247166f061ce504d9b7ccaab,"<svg xmlns=""http://www.w3.org/2000/svg"" width=..."
3,8fe82f3af89e487b31236ca829c3f071,"<svg xmlns=""http://www.w3.org/2000/svg"" width=..."
4,600464e4d92c75338462271a09b3f176,"<svg xmlns=""http://www.w3.org/2000/svg"" width=..."


In [31]:
# =========================
# CELL 16: FINAL CHECKS
# =========================
sub = pd.read_csv(SUB_PATH)

print("Submission shape:", sub.shape)
print("Columns:", sub.columns.tolist())
print("Starts with <svg rate:", sub["svg"].astype(str).str.startswith("<svg").mean())
print("Valid rate:", sub["svg"].map(simple_is_valid_svg).mean())

print("\nFirst row preview:")
print(sub.iloc[0]["svg"][:400])

Submission shape: (1000, 2)
Columns: ['id', 'svg']
Starts with <svg rate: 1.0
Valid rate: 1.0

First row preview:
<svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256"><rect x="0" y="0" width="256" height="256" fill="white"/><circle cx="128" cy="128" r="60" fill="none" stroke="black" stroke-width="8"/></svg>


In [32]:
# =========================
# CELL 17: DOWNLOAD
# =========================
try:
    from google.colab import files
    files.download(SUB_PATH)
except Exception as e:
    print("Download helper not available here.")
    print("File saved at:", SUB_PATH)
    print("Error:", e)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>